# Week 2 — Solutions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE, trustworthiness
import umap
plt.style.use("../../assets/mplstyle/course.mplstyle")
RNG = np.random.default_rng(0)

img = np.load("../data/image_embeddings.npz", allow_pickle=True)
X, y, names = img["embeddings"], img["labels"], list(img["label_names"])
text = np.load("../data/text_embeddings.npz", allow_pickle=True)
Xt, yt, names_t = text["embeddings"], text["labels"], list(text["label_names"])


## Solution 1 — t-SNE perplexity grid

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, p in zip(axes, [5, 30, 100, 500]):
    emb = TSNE(perplexity=p, init="pca", learning_rate="auto",
               random_state=0).fit_transform(X)
    tw = trustworthiness(X, emb, n_neighbors=10)
    ax.scatter(*emb.T, c=y, cmap="tab10", s=4, alpha=0.7)
    ax.set_title(f"perp={p}  T={tw:.3f}"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


**Reading.** Perplexity 30–100 produces the most legible layout for 2000 points;
perplexity 5 shatters genuine clusters into sub-clumps that *look* like fine-grained
structure but actually reflect t-SNE's bias toward small local neighbourhoods. Always
inspect a low-perplexity panel as a sanity check — if your "discovered" clusters appear
only there, they are probably an artefact.

## Solution 2 — Metric comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, m in zip(axes, ["euclidean", "cosine"]):
    emb = umap.UMAP(metric=m, n_neighbors=15, min_dist=0.1,
                    random_state=0).fit_transform(Xt)
    for c, name in enumerate(names_t):
        mask = yt == c
        ax.scatter(emb[mask, 0], emb[mask, 1], s=6, alpha=0.8, label=name)
    ax.set_title(f"metric = {m}"); ax.set_xticks([]); ax.set_yticks([])
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()


**Reading.** Sentence-transformer embeddings are L2-normalized; on the unit sphere,
Euclidean distance and cosine distance are monotonic transforms of each other, so the
layouts look similar. The difference is that **cosine** treats angular relationships as
primary, which makes the inter-cluster gaps more uniform — useful when the embedding
norms are uninformative.

## Solution 3 — Subsampling stability

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3.5))
for i, ax in enumerate(axes):
    idx = RNG.choice(len(X), size=len(X) // 2, replace=False)
    emb = umap.UMAP(n_neighbors=15, min_dist=0.1,
                    random_state=0).fit_transform(X[idx])
    ax.scatter(*emb.T, c=y[idx], cmap="tab10", s=4, alpha=0.7)
    ax.set_title(f"subsample {i + 1}"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


**Reading.** Coarse structure (the animals/vehicles super-clusters) is stable.
Fine geometric details — the orientation of the layout, the precise shape of each
cluster, the gap widths between them — are not. Interpret the *topology* (what is near
what) but not the *geometry* (how big, how far). This is the central honesty caveat
for any manifold-learning figure.